In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

In [44]:
class Value :
    def __init__(self,data,_children=(),_op='') :
        self.data=data
        self._prev=_children
        self._op=_op
        self.grad=0
        self._backward=lambda : None

    def __repr__(self) :
        return f"Value(data={self.data})"

    def __add__(self,other) :
        other = other if isinstance(other,Value) else Value(other)
        out=Value(self.data+other.data,(self,other),'+') 
        def _backward() :
            self.grad+=1*out.grad
            other.grad+=1*out.grad
        out._backward=_backward
        return out
            
    def __mul__(self,other) :
        other = other if isinstance(other,Value) else Value(other)
        out=Value(self.data*other.data,(self,other),'*')
        def _backward() :
            self.grad+=other.data*out.grad
            other.grad+=self.data*out.grad
        out._backward=_backward
        return out 
        
    def __neg__(self) :
        return self*-1
        
    def __sub__(self,other) :
        return self+(-other)

    def __rsub__(self,other):
        return other + (-self)
    
    def __pow__(self,other) :
        assert isinstance(other,(int,float)) ,"only supporting int/float powers for now"
        out=Value(self.data**other,(self,),f"**{other}")

        def _backward() :
            self.grad+=other*(self.data**(other-1))*out.grad
        out._backward=_backward
        return out
 
    def __truediv__(self,other) :
        return self*(other**-1)

    def __radd__(self,other) :
        return self+other

    def __rmul__(self,other) :
        return self*other

    def tanh(self) :
        out=Value(np.tanh(self.data),(self,),'tanh')
        def _backward() :
            self.grad+=(1-out.data**2)*out.grad
        out._backward=_backward
        return out
 
    def backward(self) :
        topo=[]
        visited=set()
        def build_topo(v) :
            if v not in visited :
                visited.add(v)
                for child in v._prev :
                    build_topo(child)
                topo.append(v)

        self.grad=1.0
        build_topo(self)
        for node in reversed(topo) :
            node._backward()

In [45]:
class Neuron :
    def __init__(self,nin) :
        self.w=[Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b=Value(random.uniform(-1,1))
                     
    def __call__(self,x) :
        act=sum((wi*xi for (wi,xi) in zip(self.w,x)),self.b)
        out=act.tanh()
        return out
        
    def parameters(self) :
        return self.w+[self.b]

    
class Layer :
    def __init__(self,nin,nout) :
        self.neurons=[Neuron(nin) for _ in range(nout)]
        
    def __call__(self,x) :
        outs=[n(x) for n in self.neurons]
        return outs[0] if len(outs)==1 else outs
        
    def parameters(self) :
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params


class MLP :
    def __init__(self,nin,nouts) :
        sz=[nin]+nouts
        self.layers=[Layer(sz[i],sz[i+1]) for i in range(len(nouts))]
    def __call__(self,x) :
        for layer in self.layers :
            x=layer(x)
        return x
    def parameters(self) :
        params = []
        for layer in self.layers :
            params.extend(layer.parameters())
        return params

In [46]:
mlp=MLP(3,[4,4,1])

In [47]:
xs=[
    [2.0,3.0,-1.0],
    [3.0,-1.0,0.5],
    [0.5,1.0,1.0],
    [1.0,1.0,-1.0],
   ]
ys=[1.0,-1.0,-1.0,1.0]
ypred=[mlp(x) for x in xs]
ypred

[Value(data=-0.8325017158814998),
 Value(data=-0.8049152375879098),
 Value(data=-0.6764614876058334),
 Value(data=-0.8237534018340337)]

In [98]:
for k in range(20) :
    #forward pass
    ypred=[mlp(x) for x in xs]
    loss=sum([(y1-y2)**2 for y1,y2 in zip(ys,ypred)])

    #backward pass
    for p in mlp.parameters() :
        p.grad=0
    loss.backward()

    #update
    for p in mlp.parameters() :
        p.data+=-0.05*p.grad
    print(k,loss.data)
    

0 0.00041471504979500586
1 0.0004142699690654803
2 0.0004138258191585237
3 0.00041338259718640775
4 0.00041294030027326707
5 0.00041249892555503265
6 0.00041205847017940185
7 0.00041161893130573407
8 0.00041118030610503013
9 0.0004107425917598532
10 0.0004103057854642737
11 0.00040986988442380523
12 0.00040943488585536375
13 0.00040900078698719326
14 0.0004085675850588088
15 0.00040813527732094866
16 0.00040770386103551744
17 0.0004072733334755153
18 0.00040684369192500535
19 0.0004064149336790307


In [99]:
ypred=[mlp(x) for x in xs]
ypred

[Value(data=0.991418507730324),
 Value(data=-0.9911313024434446),
 Value(data=-0.990562591524797),
 Value(data=0.9871693113438293)]